In [1]:
from models import BiGRUEncoder, GRUDecoder, CNNGRUDecoder
import torch
import torch.nn as nn
from typing import Dict, Any, List, Tuple
from data_loader import load_and_prepare_data, create_dataloaders
from trainer import Trainer
from inference import Inference
from utils import load_data, merge_sessions, prepare_split_dataset
from data_loader import collate_fn

In [ ]:
# Config

batch_size = 8
model = CNNGRUDecoder
learning_rate: float = 1e-3
weight_decay: float = 1e-2
checkpoint_dir: str = "BaselineGRU"
n_epochs: int = 1

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load data
print("Loading data...")
train_examples, val_examples, test_examples = load_and_prepare_data()



In [3]:
# Config

batch_size = 8
model = GRUDecoder
learning_rate: float = 1e-3
weight_decay: float = 1e-2
checkpoint_dir: str = "BaselineGRU"
n_epochs: int = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
model_params = {  "neural_dim": 512,
            "n_units": 256,
            "n_days": 2,
            "n_classes": 41,
            "rnn_dropout": 0.4,
            "input_dropout": 0.2,
            "n_layers": 3,
            "patch_size": 14,
            "patch_stride": 4,
 }

In [ ]:
# 2. Instantiate model
print("Initializing model...")
model = model(**model_params)
model.to(device)

In [ ]:

# 3. Create dataloaders
print("Creating dataloaders...")
train_loader, val_loader, test_loader = create_dataloaders(
    train_examples, val_examples, test_examples,
    model, batch_size, device
)

In [ ]:
# 4. Train model
print("Starting training...")
trainer = Trainer(
    model=model,
    device=device,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    checkpoint_dir=checkpoint_dir
)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=n_epochs,
    decode_val=True
)

In [ ]:

# 5. Load best model for inference
print("\nLoading best model for inference...")
trainer.load_checkpoint("best_model.ckpt")

In [ ]:

# 6. Generate predictions
print("Generating predictions...")
inference = Inference(model, device)
test_predictions = inference.predict(test_loader)

In [ ]:
import os
import torch

save_dir = "BaselineGRU"
os.makedirs(save_dir, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(save_dir, "gru_decoder.pth")
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GRUDecoder(**params)     # must match training params
state_dict = torch.load(
    "BaselineGRU/gru_decoder_best.pth",
    map_location=device
)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

In [ ]:
import matplotlib.pyplot as plt

train_losses = history["train_losses"]
val_losses = history["val_losses"]
val_lev_accs = history["val_lev_accs"]
val_seq_accs = history["val_seq_accs"]

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(14, 5))

# --- Losses ---
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label="Train Loss", marker="o")
plt.plot(epochs, val_losses, label="Val Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)

# --- Accuracies ---
plt.subplot(1, 2, 2)
plt.plot(epochs, val_lev_accs, label="Val Levenshtein Acc", marker="o")
plt.plot(epochs, val_seq_accs, label="Val Sequence Acc", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Metrics")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
preds[2]

In [ ]:
# LOGIT_TO_PHONEME = [
# 'BLANK',    # "BLANK" = CTC blank symbol
# 'AA', 'AE', 'AH', 'AO', 'AW',
# 'AY', 'B', 'CH', 'D', 'DH',
# 'EH', 'ER', 'EY', 'F', 'G',
# 'HH', 'IH', 'IY', 'JH', 'K',
# 'L', 'M', 'N', 'NG', 'OW',
# 'OY', 'P', 'R', 'S', 'SH',
# 'T', 'TH', 'UH', 'UW', 'V',
# 'W', 'Y', 'Z', 'ZH',
# ' | ',    # "|" = silence token
# ]

# def decode_logits_to_phonemes(indices, vocab):
#     """
#     CTC-style decoding: collapse repeats, remove blanks,
#     then map indices to phoneme strings.
#     """
#     decoded = []
#     prev = None

#     for idx in indices:
#         if idx != 0:  # 0 = BLANK
#             decoded.append(vocab[idx])
        

#     return decoded


In [ ]:
from utils import indexes_to_phonemes
indexes_to_phonemes(preds[2])